In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os
import glob
import numpy as np
import pandas as pd
import itertools
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import gridspec
from scipy.interpolate import interp1d
import matplotlib as mpl
from scipy.integrate import simpson
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
%load_ext rpy2.ipython

Load in data, prep for plotting

In [ ]:
sns.set_theme(style="ticks", context="talk", font="Arial")
plt.rcParams.update({
    "font.size": 20,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "axes.linewidth": 0.5,   
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 2,     
    "ytick.major.size": 2,
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],  # fallbacks

        # --- Keep text editable in exports ---
    "svg.fonttype": "none",   # SVG: keep text as <text> elements, not paths
    "pdf.fonttype": 42,       # PDF: embed fonts as TrueType, editable
    "ps.fonttype": 42,        # EPS: same as above
    "text.usetex": False,     # don’t force LaTeX (avoids path-conversion)
})

In [ ]:
def circular_shift_perm_test(
    df,
    event_cols=("DoorOpen","ArenaEntrance","MonsterCharge","RewardDelivered","ShelterEntrance"),
    group_cols=("Animal","Session"),
    time_col="Time",
    signal_col="Z",
    post_window=(0.0, 1.0),
    min_shift_sec=2.1,
    n_perm=2000,
    random_state=123
):
    rng = np.random.default_rng(random_state)

    need_cols = list(group_cols) + [time_col, signal_col] + list(event_cols)
    df = df.loc[:, [c for c in need_cols if c in df.columns]].copy()

    results = []
    for (gvals), gdf in df.groupby(list(group_cols), sort=False, observed=True):
        gdf = gdf.sort_values(time_col).reset_index(drop=True)

        t = gdf[time_col].to_numpy()
        z = gdf[signal_col].to_numpy()

        if len(t) < 3:
            continue
        dt = float(np.median(np.diff(t)))
        if not np.isfinite(dt) or dt <= 0:
            continue
        t0, t1 = float(t[0]), float(t[-1])
        session_len = t1 - t0
        if session_len <= min_shift_sec * 2:
            continue
        pw0, pw1 = post_window

        for ev in event_cols:
            if ev not in gdf.columns:
                continue

            ev_idx = np.flatnonzero(gdf[ev].to_numpy() == 1)
            if ev_idx.size == 0:
                continue

            peaks_obs = []
            for idx in ev_idx:
                t_ev = t[idx]
                mask = (t >= t_ev + pw0) & (t < t_ev + pw1)
                if not np.any(mask):
                    continue
                peaks_obs.append(np.max(z[mask]))

            if len(peaks_obs) == 0:
                continue

            S_obs = float(np.mean(peaks_obs))

            # build null by circularly shifting z
            null_stats = np.empty(n_perm, dtype=float)
            min_shift_steps = max(1, int(np.floor(min_shift_sec / dt)))
            n = len(z)

            lo, hi = min_shift_steps, max(min_shift_steps + 1, n - min_shift_steps)
            if hi <= lo:
                continue

            for b in range(n_perm):
                shift_steps = rng.integers(lo, hi)
                z_perm = np.roll(z, shift_steps)  # circular shift

                # recompute per-event peaks on shifted signal
                peaks_perm = []
                for idx in ev_idx:
                    t_ev = t[idx]
                    mask = (t >= t_ev + pw0) & (t < t_ev + pw1)
                    if not np.any(mask):
                        continue
                    peaks_perm.append(np.max(z_perm[mask]))

                if len(peaks_perm) == 0:
                    null_stats[b] = np.nan
                else:
                    null_stats[b] = float(np.mean(peaks_perm))

            null_stats = null_stats[np.isfinite(null_stats)]
            if null_stats.size == 0:
                continue
            p = (1.0 + np.sum(null_stats >= S_obs)) / (1.0 + null_stats.size)

            effect = S_obs - float(np.median(null_stats))

            row = {
                **{k: v for k, v in zip(group_cols, (gvals if isinstance(gvals, tuple) else (gvals,)))},
                "EventType": ev,
                "N_events": int(ev_idx.size),
                "dt_sec": dt,
                "Obs_mean_post_peak": S_obs,
                "Null_median": float(np.median(null_stats)),
                "Null_mean": float(np.mean(null_stats)),
                "Null_sd": float(np.std(null_stats, ddof=1)),
                "Effect_obs_minus_nullMedian": effect,
                "p_one_sided": p,
                "N_perm_used": int(null_stats.size),
            }
            results.append(row)

    return pd.DataFrame(results)


In [ ]:
df = pd.read_csv(r"D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\TrialDF.csv")
df['Session'] = np.where(df['Session'] == 'Reward3', 'Reward', 'Monster')
df["Session"] = pd.Categorical(df["Session"], categories=['Reward', 'Monster'], ordered=True)

successdf = df.groupby(['Animal', 'Session', 'Trial'], observed=True).first().reset_index()[['Animal', 'Session', 'Trial', 'Time to Reward', 'Time to Monster']]
successdf['Rewarded'] = np.where(successdf['Time to Reward'].notna(), 1, 0)
successdf = successdf.groupby(['Animal', 'Session'], observed=True).agg("mean").reset_index()
goodanimals  = successdf[(successdf['Session'] == 'Reward') & (successdf['Rewarded'] >= 0.8)]['Animal'].to_list()

df = pd.read_csv(r"D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\EventLockedDF.csv")
df = df[df['Animal'].isin(goodanimals)]
df['Session'] = np.where(df['Session'] == 'Reward3', 'Reward', 'Monster')
df["Session"] = pd.Categorical(df["Session"], categories=['Reward', 'Monster'], ordered=True)
df["Event"] = pd.Categorical(df["Event"], categories=['Door Open', 'Arena Entrance', 'Monster Charge', 'Reward Delivered', 'Shelter Entrance'], ordered=True)

df.rename(columns={'Region': 'Site'}, inplace=True)
df['Region'] = df['Site'].str[:2]


df['ZScore'] = df['ZScore'] - df.where(df['Time'] < 0).groupby(['Animal','Trial','Session','Event','Site'], observed=True)['ZScore'].transform('mean').fillna(0)

df = df[df['Session']=='Monster'].copy()

#sites without signal
df = df[~((df['Region']=='TS')&(df['Animal']=='Wanda'))].copy()
df = df[~((df['Site']=='TSR')&(df['Animal']=='Chester'))].copy()

Figure 3D

In [ ]:
mpl.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 9,
})

colors = ["#00B4FF", "#111111", "#F6E04A"]
CMAP = LinearSegmentedColormap.from_list("blue_black_yellow", colors, N=256)

REGIONS       = ["VS", "TS"]
SESSIONS      = [0,1]
EVENT_ORDER   = ["Door Open","Arena Entrance","Monster Charge","Reward Delivered","Shelter Entrance"]

SESSION_COLORS = {0:"C0", 1:"C3"}
LINE_YLIM      = (-1, 3)
LINE_YTICKS    = [0, 2, 4]
X_TICKS        = np.array([-2, -1, 0, 1, 2], dtype=float)
CMAP           = CMAP
VMIN, VMAX     = -3, 3.0
norm = TwoSlopeNorm(vmin=VMIN, vcenter=0, vmax=VMAX)

SESSION_NORMALIZE = {"Rewarded": "Reward"}
EVENT_NORMALIZE   = {}
REGION_NORMALIZE  = {}

def _normalize_labels(series, mapping=None):
    s = series.astype(str).str.strip().str.replace(".", " ", regex=False)
    if mapping:
        s = s.replace(mapping)
    return s

def _blank(ax):
    ax.set_facecolor("white")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel(""); ax.set_ylabel("")

def _sync_heat_xticks(ax, heat_cols, xticks):
    cols = pd.to_numeric(pd.Index(heat_cols), errors='coerce').to_numpy()
    keep = [t for t in xticks if np.isfinite(cols).all() and cols.min() <= t <= cols.max()]
    if not keep:
        ax.set_xticks([]); return
    pos = [int(np.nanargmin(np.abs(cols - t))) for t in keep]
    ax.set_xticks(pos); ax.set_xticklabels([f"{t:g}" for t in keep], rotation=0)

def line_overlay_sessions(ax, df_cell, region):
    drew_any = False

    for sess in SESSIONS:
        sub = df_cell[(df_cell["Outcome"] == sess) & (df_cell["Region"] == region)]
        if sub.empty:
            continue

        per_animal = (sub.groupby(["Animal", "Time"], observed=True)["ZScore"]
                        .mean()
                        .reset_index())

        agg = (per_animal.groupby("Time", observed=True)["ZScore"]
                         .agg(["mean", "std", "count"])
                         .reset_index()
                         .sort_values("Time"))
        sem = agg["std"] / np.sqrt(agg["count"])
        mu  = agg["mean"]
        t   = agg["Time"]

        color = SESSION_COLORS.get(sess, None)

        ax.plot(t, mu, label=sess, color=color, lw=1.8)
        ax.fill_between(t, mu - sem, mu + sem, alpha=0.25, edgecolor="none", color=color)

        drew_any = True
    if not drew_any:
        _blank(ax); return
    ax.axvline(0, color='k', lw=0.9, ls='--')
    ax.axhline(0, color='0.6', lw=0.9, ls='--')
    ax.set_xlim(X_TICKS.min(), X_TICKS.max())
    ax.set_ylim(*LINE_YLIM)
    ax.set_yticks(LINE_YTICKS)
    ax.set_xticks(X_TICKS); ax.set_xticklabels([f"{t:g}" for t in X_TICKS])
    ax.tick_params(length=3, width=0.8)
    ax.axvline(0, color='k', lw=0.9, ls='--')
    ax.axhline(0, color='0.6', lw=0.9, ls='--')
    ax.set_xlim(X_TICKS.min(), X_TICKS.max())
    ax.set_ylim(*LINE_YLIM)
    ax.set_yticks(LINE_YTICKS)
    ax.set_xticks(X_TICKS); ax.set_xticklabels([f"{t:g}" for t in X_TICKS])
    ax.tick_params(length=3, width=0.8)

def heatmap_for(ax, df_cell, session, region):
    sub = df_cell[(df_cell["Outcome"] == session) & (df_cell["Region"] == region)]
    if sub.empty:
        _blank(ax); return False
    order_idx = (sub[(sub["Time"] >= 0) & (sub["Time"] <= 1)]
                 .groupby("Animal", observed=True)["ZScore"]
                 .mean().sort_values(ascending=False).index.tolist())
    H = (sub.groupby(["Animal","Time"], observed=True)["ZScore"]
           .mean().unstack().sort_index(axis=1))
    keep   = [a for a in order_idx if a in H.index]
    others = [a for a in H.index if a not in keep]
    H = H.reindex(keep + others)
    sns.heatmap(H, ax=ax, cmap=CMAP, vmin=VMIN, vmax=VMAX, cbar=False)
    ax.set_ylabel(""); ax.set_yticks([]); ax.tick_params(axis='y', length=0)
    _sync_heat_xticks(ax, H.columns, X_TICKS)
    return True

plot_df = df.copy()
plot_df = plot_df[
    plot_df["Region"].isin(REGIONS) &
    plot_df["Outcome"].isin(SESSIONS) &
    plot_df["Event"].isin(EVENT_ORDER)
].copy()
plot_df["Region"]  = pd.Categorical(plot_df["Region"],  categories=REGIONS,     ordered=True)
plot_df["Outcome"] = pd.Categorical(plot_df["Outcome"], categories=SESSIONS,    ordered=True)
plot_df["Event"]   = pd.Categorical(plot_df["Event"],   categories=EVENT_ORDER, ordered=True)

n_evt = len(EVENT_ORDER)
fig = plt.figure(figsize=(2.9*n_evt + 2.0, 5.8))
outer_all = gridspec.GridSpec(nrows=1, ncols=1, height_ratios=[1.0], hspace=0.0)

col_axes_top = {}  

matrix = gridspec.GridSpecFromSubplotSpec(
    nrows=1, ncols=n_evt + 1, subplot_spec=outer_all[0],
    height_ratios=[1], width_ratios=[1]*n_evt + [0.06],
    hspace=0.4, wspace=0.28
)
sm = mpl.cm.ScalarMappable(cmap=CMAP, norm=norm); sm.set_array([])

legend_handles, legend_labels = None, None

for c, event in enumerate(EVENT_ORDER):
    cell_df = plot_df[plot_df["Event"] == event]

    cell_gs = gridspec.GridSpecFromSubplotSpec(
        nrows=3, ncols=2, subplot_spec=matrix[0, c],
        height_ratios=[1.1, 1.0, 1.0],
        width_ratios=[1, 1],
        hspace=0.08, wspace=0.10
    )
    ax_vs = fig.add_subplot(cell_gs[0, 0])
    ax_ts = fig.add_subplot(cell_gs[0, 1])
    line_overlay_sessions(ax_vs, cell_df, "VS")
    line_overlay_sessions(ax_ts, cell_df, "TS")

    if c > 0:
        ax_vs.set_yticklabels([]); ax_vs.set_ylabel("")
    ax_ts.set_yticklabels([]); ax_ts.set_ylabel("")
    ax_vs.set_xticklabels([]); ax_ts.set_xticklabels([])

    if c == 0:
        legend_handles, legend_labels = ax_vs.get_legend_handles_labels()
    for ax in (ax_vs, ax_ts):
        leg = ax.get_legend()
        if leg: leg.remove()

    ax_r_vs = fig.add_subplot(cell_gs[1, 0]); heatmap_for(ax_r_vs, cell_df, 1,  "VS")
    ax_r_ts = fig.add_subplot(cell_gs[1, 1]); heatmap_for(ax_r_ts, cell_df, 1,  "TS")
    ax_m_vs = fig.add_subplot(cell_gs[2, 0]); heatmap_for(ax_m_vs, cell_df, 0, "VS")
    ax_m_ts = fig.add_subplot(cell_gs[2, 1]); heatmap_for(ax_m_ts, cell_df, 0, "TS")

    for ax in (ax_r_vs, ax_r_ts):
        ax.set_xlabel(""); ax.set_xticks([]); ax.set_xticklabels([])
        ax.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)

    ax_m_vs.set_xlabel("Time (s)"); ax_m_ts.set_xlabel("Time (s)")

    col_axes_top[event] = (ax_vs, ax_ts)

cax_matrix = fig.add_subplot(matrix[:, -1])
cb = plt.colorbar(sm, cax=cax_matrix)
cax_matrix.set_ylabel("ZScore", rotation=90, labelpad=8)
cax_matrix.tick_params(labelsize=8)

if legend_handles:
    fig.legend(legend_handles, legend_labels, loc="upper center",
               ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.975), columnspacing=1.3)

fig.text(0.005, 0.67, "Z Score", rotation=90, va="center", ha="left", fontsize=10)

plt.tight_layout(rect=[0.02, 0.03, 0.98, 0.955])
fig.canvas.draw()

EVENT_FS   = 12.0
REGION_FS  = 10.0
PAD_EVT    = 0.03
PAD_TAG    = 0.004

for event in EVENT_ORDER:
    if event in col_axes_top:
        ax_vs, ax_ts = col_axes_top[event]
        bb_vs = ax_vs.get_position(); bb_ts = ax_ts.get_position()
        col_center = 0.5 * (bb_vs.x0 + bb_ts.x1)
        y_top      = max(bb_vs.y1, bb_ts.y1)
        plt.figtext(col_center, y_top + PAD_EVT, event,
                    ha="center", va="bottom", fontsize=EVENT_FS, weight="bold")
        vs_center = 0.5 * (bb_vs.x0 + bb_vs.x1)
        ts_center = 0.5 * (bb_ts.x0 + bb_ts.x1)
        plt.figtext(vs_center, y_top + PAD_TAG, "VS",
                    ha="center", va="bottom", fontsize=REGION_FS)
        plt.figtext(ts_center, y_top + PAD_TAG, "TS",
                    ha="center", va="bottom", fontsize=REGION_FS)

fig.suptitle("", fontsize=14, fontweight="bold", y=0.993)
plt.savefig(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\OutcomeEventLocked.svg')
plt.show()


In [ ]:
df = pd.read_csv(r"D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\FullTraceDF.csv")

df = df[df['Session'] == 'Monster_1'].copy()
df['Trial'] = df.groupby(["Animal","Session","Site","Region"])['DoorOpen'].cumsum()
df['Trial'] = np.where(df['Trial']==0, 1, df['Trial'])
df["Outcome"] = (
    df.groupby(["Animal", "Session", "Site", "Region", "Trial"])["RewardDelivered"]
      .transform(lambda x: "Unrewarded" if x.sum() == 0 else "Rewarded")
)

res = circular_shift_perm_test(df, n_perm=2000, group_cols=("Animal","Session","Site","Region", "Outcome"))

In [ ]:
%%R -i res
library(lme4)
library(emmeans)

test <- lmer(Effect_obs_minus_nullMedian~EventType*Outcome*Region+(1|Animal) + (1|Animal:Site), data=res)

emm <- emmeans(test, ~ EventType*Outcome | Region)
print(test(emm, adjust = "bonferroni"))


test_res <- test(emm, adjust = "bonferroni")
test_df  <- as.data.frame(summary(test_res))

print(test_df[, c("Region", "EventType", "Outcome", "t.ratio", "p.value")])

emm <- emmeans(test, ~ Outcome | EventType*Region)
print(test(pairs(emm, adjust = "bonferroni")))

emm <- emmeans(test, ~ Outcome*Region | EventType)
print(test(pairs(emm, adjust = "bonferroni")))